In [ ]:
#inputs: prarie vole and mouse SAM
#outputs: mapped subclasses between prarie vole and mouse

In [193]:
sam1=SAM()
sam1.load_data('Active_SAM_joined/SAM_MO_soupx_plus5_cleaned_03122025.h5ad')

In [194]:
!pip install anndata==0.8.0

In [195]:
!pip install loompy

In [196]:
from samalg import SAM
import scanpy as sc
import statistics
import matplotlib.pyplot as plt
import seaborn as sns
import random
import pandas as pd
import matplotlib.colors
import scipy
import numpy as np
import sklearn.metrics as metrics
from scipy import sparse
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import loompy
import time
import pickle

In [197]:
sam1.adata.obs

,orig.ident,nCount_RNA,nFeature_RNA,n_counts,n_genes,key,subclass_id_label_mapping,subclass_id_label_lc,leiden_clusters,subclass_id_label_mapping_nounlabeled,...,SCT_snn_res.0.8,seurat_clusters,SCT_snn_res.5,eq_subclass,eq_subclass_lc,eq_subclass_frac,eq_subclass_nounlabeled,NN,ss_subclass,ss_subclass_nounlabeled
AAACCCATCTCAAAGC,MO,11772.0,3844,11772.0,3844,Run12_sample1,Unlabeled,131,61,mo_15,...,10,15,15,Unlabeled,86,1.0,mo_26,Unlabeled,Unlabeled,mo_18
AAACGAACAGCACGAA,MO,4877.0,2561,4877.0,2561,Run12_sample1,Unlabeled,290,78,mo_10,...,5,105,105,Unlabeled,240,1.0,mo_7,Unlabeled,Unlabeled,mo_6
AAACGAAGTCGGCACT,MO,5221.0,2859,5221.0,2859,Run12_sample1,108 ARH-PVp Tbx3 Gaba,27,5,108 ARH-PVp Tbx3 Gaba,...,9,14,14,108 ARH-PVp Tbx3 Gaba,16,1.0,108 ARH-PVp Tbx3 Gaba,Neuron,108 ARH-PVp Tbx3 Gaba,108 ARH-PVp Tbx3 Gaba
AAACGAATCCACCCTA,MO,7922.0,3360,7922.0,3360,Run12_sample1,Unlabeled,529,103,mo_9,...,18,87,87,Unlabeled,527,1.0,mo_47,Unlabeled,Unlabeled,mo_38
AAACGAATCCCAGCGA,MO,7020.0,3204,7020.0,3204,Run12_sample1,128 VMH Fezf1 Glut,104,3,128 VMH Fezf1 Glut,...,2,16,16,128 VMH Fezf1 Glut,333,1.0,128 VMH Fezf1 Glut,Neuron,128 VMH Fezf1 Glut,128 VMH Fezf1 Glut
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGGTACTAACC,MO,5496.0,2569,5496.0,2569,Run12_sample12,057 NDB-SI-MA-STRv Lhx8 Gaba,217,30,057 NDB-SI-MA-STRv Lhx8 Gaba,...,12,26,26,057 NDB-SI-MA-STRv Lhx8 Gaba,156,1.0,057 NDB-SI-MA-STRv Lhx8 Gaba,Neuron,057 NDB-SI-MA-STRv Lhx8 Gaba,057 NDB-SI-MA-STRv Lhx8 Gaba
TTTGTTGGTCACAATC-1,MO,2974.0,1804,2974.0,1804,Run12_sample12,089 PVR Six3 Sox3 Gaba,204,23,089 PVR Six3 Sox3 Gaba,...,6,52,52,089 PVR Six3 Sox3 Gaba,119,1.0,089 PVR Six3 Sox3 Gaba,Neuron,089 PVR Six3 Sox3 Gaba,089 PVR Six3 Sox3 Gaba
TTTGTTGTCCAACCAA,MO,5601.0,2608,5601.0,2608,Run12_sample12,Unlabeled,211,28,mo_11,...,6,56,56,Unlabeled,265,1.0,mo_6,Unlabeled,Unlabeled,mo_4
TTTGTTGTCCCTGTTG,MO,6209.0,2759,6209.0,2759,Run12_sample12,Unlabeled,478,57,mo_4,...,7,33,33,Unlabeled,522,1.0,mo_8,Unlabeled,Unlabeled,mo_7


In [198]:
sam1.adata.obs = sam1.adata.obs.drop(['subclass_id_label_mapping','subclass_id_label_lc','leiden_clusters','subclass_id_label_mapping_nounlabeled','neurotransmitter','region_label','subclass_id_label_reduced_mapping','subclass_id_label_reduced_lc','subclass_id_label_reduced_mapping_nounlabeled'], axis = 1)

In [199]:
sam1.adata.obs = sam1.adata.obs.drop(['eq_subclass','eq_subclass_lc','eq_subclass_frac'],axis=1)

In [200]:
sam1.adata.obs = sam1.adata.obs.drop(['leiden_clusters_formarkers','nCount_SCT','nFeature_SCT','SCT_snn_res.0.8','seurat_clusters'], axis = 1)

In [ ]:
org = 'mo'
ref = 'mg'
dat = sc.read_h5ad('SAM_Allen_Insitute_NN_subclass_250.h5ad')

#make sure that the obs and var names are unique
sam.adata.obs_names_make_unique()
sam.adata.var_names_make_unique()

sam1.adata.obs_names_make_unique()
sam1.adata.var_names_make_unique()

sams = {ref:sam,org:sam1}

sm = SAMAP(
    sams,
    f_maps = 'BLASTMAPPING/maps/MO_maps/',
)
sm.run(pairwise=True)

level = 'subclass_id_label'
t0 = time.time()
spline = []
num_cell_types = []

#range of leiden clustering resolutions 
r = range(5,80,5)
for reso in r:
    sm.sams[org].clustering(param=reso)

    #get mapping between mouse and org
    keys = {ref:level,org:'leiden_clusters'}
    D,MappingTable = get_mapping_scores(sm,keys)
    lim_MappingTable = MappingTable.filter(like=org + '_')
    lim_MappingTable = lim_MappingTable[lim_MappingTable.index.str.contains(ref)]

    #find the best mapping that has more than 25 cells and greater than .2 alignment score 
    mapping_dict = {}
    for item in lim_MappingTable:
        len_item = len(sm.sams[org].adata[sm.sams[org].adata.obs['leiden_clusters'] == int(item[3:])])
        if len_item > 25 and max(lim_MappingTable[item]) > .2:
            mapping_dict[item] = str(lim_MappingTable[item].idxmax())
        else:
            mapping_dict[item] = ref + '_Unlabeled'

    #saving the new mappings and leiden clusters
    new_mapping = []
    for item in sm.sams[org].adata.obs['leiden_clusters']:
        new_mapping.append(mapping_dict[org + '_' + str(item)][3:])

    sm.sams[org].adata.obs['temp_mapping_' + level + '_reso_' + str(reso)] = new_mapping
    sm.sams[org].adata.obs['temp_mapping_lc_reso_' + str(reso)] = sm.sams[org].adata.obs['leiden_clusters']
    num_cell_types.append(sm.sams[org].adata.obs['temp_mapping_' + level + '_reso_' + str(reso)].nunique())
    t1 = time.time()
    print('finished mapping ' + str((reso/5)*(100/25)) + ' percent in ' + str(t1-t0) + ' seconds')

#get a function for the curve and plot to see how well it matches data
x = [i for i in r]
func = np.polyfit(np.log(x), num_cell_types, 1)
plt.scatter(x, num_cell_types)

#pick the first resolution after which adding another 1 to the resolution will likely add only 1 new cell type
val = 0
best_reso = 0
for i in range(5,r[-1],1):
    if np.log(i) *func[0] + func[1] > val + 1:
        val = np.log(i) *func[0] + func[1]
        best_reso = i
    else:
        break
print(best_reso)

#Set best resolution manually
best_reso = 5

#remove the temporary mappings and keep the actual mapping
for i in r:
    if i != best_reso:
        print(i)
        sm.sams[org].adata.obs = sm.sams[org].adata.obs.drop('temp_mapping_' + level + '_reso_' + str(i), axis = 1)
        sm.sams[org].adata.obs = sm.sams[org].adata.obs.drop('temp_mapping_lc_reso_' + str(i), axis = 1)
    else:
        sm.sams[org].adata.obs[level + '_mapping'] = sm.sams[org].adata.obs['temp_mapping_' + level + '_reso_' + str(i)]
        sm.sams[org].adata.obs[level + '_lc'] = sm.sams[org].adata.obs['temp_mapping_lc_reso_' + str(i)]
        sm.sams[org].adata.obs = sm.sams[org].adata.obs.drop('temp_mapping_' + level + '_reso_' + str(i), axis = 1)
        sm.sams[org].adata.obs = sm.sams[org].adata.obs.drop('temp_mapping_lc_reso_' + str(i), axis = 1)
sm.sams['mo'].adata.obs[level + '_mapping'].to_csv('Active_SAMap_Joined/MO_metadata_soupxplus5_cleaned_03122025_subclass_250_0.csv')

In [ ]:
for indexer in range(1,30):
    org = 'mo'
    ref = 'mg'
    dat = sc.read_loom('Subclustering/subset_Allen_institute_Full_subclass_test_250_'+str(i)+'.loom')
    dat.obs_names = dat.obs['obs_names']
    dat.var_names = list(dat.var['x'])
    
    sam=SAM(dat)
    sam.preprocess_data()
    sam.run()
    
    #make sure that the obs and var names are unique
    sam.adata.obs_names_make_unique()
    sam.adata.var_names_make_unique()

    sam1.adata.obs_names_make_unique()
    sam1.adata.var_names_make_unique()
    
    sams = {ref:sam,org:sam1}

    sm = SAMAP(
        sams,
        f_maps = 'BLASTMAPPING/maps/MO_maps/',
    )
    sm.run(pairwise=True)
    save_samap(sm,'Active_SAMap_Joined/sm_Allen_Full_mo_soupxplus5_cleaned_03122025_subclass_250_'+str(indexer)+'.pkl'
    
    level = 'subclass_id_label'
    t0 = time.time()
    spline = []
    num_cell_types = []
    best_reso = ###manually set best resolution
    r = range(best_reso,best_reso + 5,5)
    

    #range of leiden clustering resolutions 
    for reso in r:
        sm.sams[org].clustering(param=reso)

        #get mapping between mouse and org
        keys = {ref:level,org:'leiden_clusters'}
        D,MappingTable = get_mapping_scores(sm,keys)
        lim_MappingTable = MappingTable.filter(like=org + '_')
        lim_MappingTable = lim_MappingTable[lim_MappingTable.index.str.contains(ref)]

        #find the best mapping that has more than 25 cells and greater than .2 alignment score 
        mapping_dict = {}
        for item in lim_MappingTable:
            len_item = len(sm.sams[org].adata[sm.sams[org].adata.obs['leiden_clusters'] == int(item[3:])])
            if len_item > 25 and max(lim_MappingTable[item]) > .2:
                mapping_dict[item] = str(lim_MappingTable[item].idxmax())
            else:
                mapping_dict[item] = ref + '_Unlabeled'

        #saving the new mappings and leiden clusters
        new_mapping = []
        for item in sm.sams[org].adata.obs['leiden_clusters']:
            new_mapping.append(mapping_dict[org + '_' + str(item)][3:])

        sm.sams[org].adata.obs['temp_mapping_' + level + '_reso_' + str(reso)] = new_mapping
        sm.sams[org].adata.obs['temp_mapping_lc_reso_' + str(reso)] = sm.sams[org].adata.obs['leiden_clusters']
        num_cell_types.append(sm.sams[org].adata.obs['temp_mapping_' + level + '_reso_' + str(reso)].nunique())
        t1 = time.time()
        print('finished mapping ' + str((reso/5)*(100/25)) + ' percent in ' + str(t1-t0) + ' seconds')

    #remove the temporary mappings and keep the actual mapping
    for i in r:
        if i != best_reso:
            print(i)
            sm.sams[org].adata.obs = sm.sams[org].adata.obs.drop('temp_mapping_' + level + '_reso_' + str(i), axis = 1)
            sm.sams[org].adata.obs = sm.sams[org].adata.obs.drop('temp_mapping_lc_reso_' + str(i), axis = 1)
        else:
            sm.sams[org].adata.obs[level + '_mapping'] = sm.sams[org].adata.obs['temp_mapping_' + level + '_reso_' + str(i)]
            sm.sams[org].adata.obs[level + '_lc'] = sm.sams[org].adata.obs['temp_mapping_lc_reso_' + str(i)]
            sm.sams[org].adata.obs = sm.sams[org].adata.obs.drop('temp_mapping_' + level + '_reso_' + str(i), axis = 1)
            sm.sams[org].adata.obs = sm.sams[org].adata.obs.drop('temp_mapping_lc_reso_' + str(i), axis = 1)
            sm.sams['mo'].adata.obs.to_csv('Active_SAMap_Joined/MO_metadata_soupxplus5_cleaned_03122025_subclass_250_'+str(indexer)+'.csv')